# lm_forge — Pretrain 160M Base Model (Colab T4)

This notebook pretrains a 160M parameter model on:
- **Phase 1:** `HuggingFaceFW/fineweb-edu` (general knowledge)
- **Phase 2:** `bigcode/the-stack-v2` (code)

**Hardware:** Google Colab T4 (16GB VRAM)  
**Strategy:** Zero-Bloat Hub Checkpointing (Colab Nomad Pattern)

**Resumption:** This system automatically pulls the latest `checkpoint-XXXX` from your Hugging Face Hub `checkpoints` branch, ensuring seamless resumption across session disconnects.

## 1. Setup & Dependencies

In [ ]:
# Clone repo (skip if already cloned)
import os
if not os.path.exists('lm_forge'):
    !git clone https://github.com/Kush26/lm_forge.git
%cd lm_forge

# Install dependencies
!pip install -q torch transformers datasets huggingface_hub>=0.28.0 safetensors accelerate wandb pyyaml

In [ ]:
# Login to HF (Required for Checkpoint Sync)
from google.colab import userdata
import os

try:
    # Recommended: Add 'HF_TOKEN' to Colab Secrets (key icon on the left)
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from Colab Secrets.")
except:
    print("HF_TOKEN not found in Secrets. Please login manually.")
    from huggingface_hub import notebook_login
    notebook_login()

## 2. Phase 1 — General Knowledge (fineweb-edu)

**Just re-run this cell if your session disconnects.** It will:
1. Auto-pull the latest checkpoint from your Hub `checkpoints` branch.
2. Resume training using T4-optimized settings (Batch=8, GradAccum=32, FP16).
3. Auto-push a full state backup (weights + optimizer) every 1000 steps.

In [ ]:
!python experiments/pretrain_base/train.py --phase 1 --t4

## 3. Phase 2 — Code (the-stack-v2)

Run this **after Phase 1 is complete.** It will pull from the `final/` folder created in Phase 1.

In [ ]:
!python experiments/pretrain_base/train.py --phase 2 --t4

## 4. Monitoring

Training logs are pushed to Weights & Biases (W&B) and the `checkpoints` folder locally. Check your Hub repo's `checkpoints` branch to see the remote backups.